# Chapter 1.5: Mixture Models with Hidden Variables

**Key Idea**: Neural neurons can be in different states. Mixture models help us discover these latent states from data.

**Topics**: Hidden variables, Gaussian Mixture Models, Expectation-Maximization preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, gamma, norm
import seaborn as sns
from scipy.special import logsumexp

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

print('Ready for Mixture Models!')

## Motivation: The Hidden States Problem

### Observed: Spike Counts
```
Time:   [1s, 2s, 3s, 4s, 5s, 6s, 7s, 8s, 9s, 10s]
Spikes: [2,  5,  1,  4,  3,  6,  2,  5,  4,  3]
```

### Hidden: What state was neuron in?
```
State:  [L,  H,  L,  H,  L,  H,  L,  H,  H,  L]  ← UNKNOWN!
```
L = Low (lambda_L ≈ 2)
H = High (lambda_H ≈ 5)

### Challenge
Given only spike counts, infer the hidden states

## Part 1: Mixture Model Formula

### Single Observation

For observation x_t:

```
p(x_t) = sum over hidden states of:
         P(state) * P(x_t | state)
```

**Two-state example**:
```
p(x_t) = pi_L * Poisson(x_t | lambda_L) + pi_H * Poisson(x_t | lambda_H)

where:
  pi_L = probability of Low state
  pi_H = probability of High state
  pi_L + pi_H = 1
```

### Visualization

The mixture is a WEIGHTED SUM of two distributions:
```
         Mixture (what we observe)
            /\
           /  \
        50% Low + 50% High
```

## Part 2: Generating Data from a Mixture

In [ ]:
# True parameters
lambda_low = 2.0
lambda_high = 5.0
pi_low = 0.5     # 50% of time in low state
pi_high = 0.5    # 50% of time in high state

T = 200  # Total observations

# Generate hidden states
hidden_states = np.random.binomial(n=1, p=pi_high, size=T)
# 0 = Low state, 1 = High state

# Generate observations based on hidden states
observations = np.zeros(T)
for t in range(T):
    if hidden_states[t] == 0:  # Low state
        observations[t] = np.random.poisson(lam=lambda_low)
    else:  # High state
        observations[t] = np.random.poisson(lam=lambda_high)

print('='*60)
print('Mixture Model Data Generation')
print('='*60)
print(f'\nTrue Parameters:')
print(f'  Low state:  lambda_L = {lambda_low}, prob = {pi_low}')
print(f'  High state: lambda_H = {lambda_high}, prob = {pi_high}')

print(f'\nGenerated Data:')
print(f'  Total observations: {T}')
print(f'  Actual Low states: {np.sum(hidden_states == 0)} ({np.mean(hidden_states == 0):.1%})')
print(f'  Actual High states: {np.sum(hidden_states == 1)} ({np.mean(hidden_states == 1):.1%})')

print(f'\nFirst 20 observations:')
print(f'  Spikes: {observations[:20].astype(int)}')
print(f'  State:  {["L" if s == 0 else "H" for s in hidden_states[:20]]}')

print(f'\nObserved spike statistics:')
print(f'  Mean: {np.mean(observations):.2f}')
print(f'  Std:  {np.std(observations):.2f}')
print(f'  Expected from mixture: {pi_low * lambda_low + pi_high * lambda_high:.2f}')

## Part 3: The Problem - We Don't Know Hidden States

In reality, we ONLY see the spikes, NOT the hidden states!

Given observations, can we:
1. Recover the hidden states?
2. Learn the parameters (lambda_L, lambda_H, pi_L, pi_H)?

In [ ]:
# Visualize: What we observe vs what's really happening
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

t_range = range(100)  # First 100 points

# Plot 1: Hidden states (TRUE - we don't see this)
ax = axes[0]
colors = ['blue' if s == 0 else 'red' for s in hidden_states[t_range]]
ax.scatter(t_range, hidden_states[t_range], c=colors, s=100, alpha=0.6)
ax.axhline(0, color='blue', linestyle='--', alpha=0.3, label='Low state')
ax.axhline(1, color='red', linestyle='--', alpha=0.3, label='High state')
ax.set_ylabel('Hidden State')
ax.set_title('(a) TRUE Hidden States (We DO NOT see this in real life!)')
ax.set_ylim([-0.2, 1.2])
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Observations colored by true state
ax = axes[1]
ax.bar(t_range, observations[t_range], color=colors, alpha=0.7, edgecolor='black')
ax.axhline(lambda_low, color='blue', linestyle='--', linewidth=2, label='lambda_L (low state)')
ax.axhline(lambda_high, color='red', linestyle='--', linewidth=2, label='lambda_H (high state)')
ax.set_ylabel('Spike Count')
ax.set_title('(b) Observations, colored by TRUE state (for visualization only)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: What we really see (just the numbers)
ax = axes[2]
ax.bar(t_range, observations[t_range], color='gray', alpha=0.7, edgecolor='black')
ax.axhline(np.mean(observations), color='black', linestyle='--', linewidth=2, 
           label=f'Observed mean = {np.mean(observations):.2f}')
ax.set_xlabel('Time')
ax.set_ylabel('Spike Count')
ax.set_title('(c) What We ACTUALLY See (No color coding, no state info!)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('KEY INSIGHT: We only see plot (c), not (a) or (b)!')
print('Task: Recover hidden states from mixture observations')

## Part 4: Posterior Over Hidden States

### Question
Given observation x_t, what's the probability it came from each state?

### Answer: Bayes Rule

```
P(state | x_t) = P(x_t | state) * P(state) / P(x_t)

where:
P(x_t) = sum over states: P(x_t | state) * P(state)  (mixture formula)
```

### For Two States

```
P(Low | x_t) = P(x_t | Low) * pi_L / [P(x_t | Low)*pi_L + P(x_t | High)*pi_H]

P(High | x_t) = P(x_t | High) * pi_H / [P(x_t | Low)*pi_L + P(x_t | High)*pi_H]
```

### Intuition

- If x_t = 1 (low count) → more likely from Low state
- If x_t = 7 (high count) → more likely from High state
- If x_t = 3 (ambiguous) → both states are possible

In [ ]:
def mixture_posterior(x, lambda_low, lambda_high, pi_low, pi_high):
    """
    Compute posterior probability of hidden states given observation
    
    Returns:
      p_low: P(Low | x)
      p_high: P(High | x)
    """
    # Likelihood of observation under each state
    ll_low = poisson.pmf(x, mu=lambda_low)
    ll_high = poisson.pmf(x, mu=lambda_high)
    
    # Posterior (Bayes rule)
    numerator_low = ll_low * pi_low
    numerator_high = ll_high * pi_high
    denominator = numerator_low + numerator_high
    
    p_low = numerator_low / denominator
    p_high = numerator_high / denominator
    
    return p_low, p_high

# Test with different observations
test_counts = [0, 1, 2, 3, 4, 5, 6, 7, 8]

print('='*60)
print('Posterior Over Hidden States')
print('='*60)
print(f'\nTrue parameters:')
print(f'  lambda_L = {lambda_low}, lambda_H = {lambda_high}')
print(f'  pi_L = {pi_low}, pi_H = {pi_high}')

print(f'\nGiven different spike counts, probability of each state:')
print(f'\n{"Count":<8} {"P(Low|x)":<12} {"P(High|x)":<12} {"Most Likely":<15}')
print('-' * 50)

for x in test_counts:
    p_low, p_high = mixture_posterior(x, lambda_low, lambda_high, pi_low, pi_high)
    most_likely = 'Low' if p_low > p_high else 'High'
    print(f'{x:<8} {p_low:<12.4f} {p_high:<12.4f} {most_likely:<15}')

## Part 5: Visualizing Posteriors

In [ ]:
# Compute posteriors for all observations
posteriors = np.array([mixture_posterior(obs, lambda_low, lambda_high, pi_low, pi_high) 
                       for obs in observations])

p_low_given_x = posteriors[:, 0]
p_high_given_x = posteriors[:, 1]

# Predicted states (argmax of posterior)
predicted_states = (p_high_given_x > 0.5).astype(int)

# Calculate accuracy
accuracy = np.mean(predicted_states == hidden_states)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

t_range = range(100)

# Plot 1: Posterior probabilities over time
ax = axes[0]
ax.fill_between(t_range, 0, p_low_given_x[t_range], alpha=0.5, 
                 color='blue', label='P(Low | x_t)')
ax.fill_between(t_range, p_low_given_x[t_range], 1, alpha=0.5, 
                 color='red', label='P(High | x_t)')

# Overlay true states as background
for t in t_range:
    if hidden_states[t] == 0:
        ax.axvspan(t-0.4, t+0.4, alpha=0.1, color='blue')
    else:
        ax.axvspan(t-0.4, t+0.4, alpha=0.1, color='red')

ax.set_ylabel('Posterior Probability')
ax.set_title('(a) Posterior Probabilities: P(Low|x_t) and P(High|x_t)')
ax.set_ylim([0, 1])
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Decision boundary
ax = axes[1]
colors_predicted = ['blue' if s == 0 else 'red' for s in predicted_states[t_range]]
colors_true = ['blue' if s == 0 else 'red' for s in hidden_states[t_range]]
markers = ['o' if predicted_states[t] == hidden_states[t] else 'x' for t in t_range]

for t in t_range:
    marker = 'o' if predicted_states[t] == hidden_states[t] else 'X'
    ax.scatter(t, observations[t], c=colors_true[t-t_range[0]], s=100, 
              marker=marker, alpha=0.7, edgecolors='black', linewidth=1.5)

ax.axhline(lambda_low, color='blue', linestyle='--', linewidth=2, label='lambda_L')
ax.axhline(lambda_high, color='red', linestyle='--', linewidth=2, label='lambda_H')
ax.axhline((lambda_low + lambda_high) / 2, color='gray', linestyle=':', 
           linewidth=2, label='Decision boundary')

ax.set_xlabel('Time')
ax.set_ylabel('Spike Count')
ax.set_title(f'(b) State Inference: True state color, Circles=Correct, X=Errors, Accuracy={accuracy:.1%}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'\nInference Accuracy: {accuracy:.1%}')
print(f'Correct predictions: {np.sum(predicted_states == hidden_states)} / {T}')

## Part 6: The Learning Problem

### What if we DON'T know the true parameters?

In real data, we need to learn:
- lambda_L (low state spike rate)
- lambda_H (high state spike rate)
- pi_L, pi_H (state probabilities)

### Chicken-and-Egg Problem

```
To learn parameters, we need hidden states
To infer hidden states, we need parameters
                  ↓
            CIRCULAR DEPENDENCY!
```

### Solution: EM Algorithm

Expectation-Maximization iteratively:
1. E-step: Infer hidden states (use current parameter estimates)
2. M-step: Update parameters (use inferred hidden states)
3. Repeat until convergence

(EM Algorithm is Chapter 1.6)

## Part 7: Mixture of Gaussians (Bonus)

The same ideas work for Gaussian distributions too!

```
p(x) = pi_1 * N(x | mu_1, sigma_1) + pi_2 * N(x | mu_2, sigma_2)
```

In [ ]:
# Mixture of Gaussians example
mu_1, sigma_1, pi_1 = -2, 0.5, 0.4
mu_2, sigma_2, pi_2 = 2, 0.5, 0.6

# Generate data from Gaussian mixture
N = 1000
z = np.random.binomial(1, pi_2, N)  # Hidden state
x_gaussian = np.where(z == 0, 
                      np.random.normal(mu_1, sigma_1, N),
                      np.random.normal(mu_2, sigma_2, N))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Component distributions
ax = axes[0]
x_range = np.linspace(-4, 4, 1000)

ax.plot(x_range, pi_1 * norm.pdf(x_range, mu_1, sigma_1), 'b-', linewidth=2.5, label='Component 1')
ax.plot(x_range, pi_2 * norm.pdf(x_range, mu_2, sigma_2), 'r-', linewidth=2.5, label='Component 2')
ax.plot(x_range, pi_1 * norm.pdf(x_range, mu_1, sigma_1) + 
                  pi_2 * norm.pdf(x_range, mu_2, sigma_2), 'k--', linewidth=2.5, label='Mixture')

ax.set_xlabel('x')
ax.set_ylabel('Probability')
ax.set_title('Mixture of Two Gaussians')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Sample histogram
ax = axes[1]
ax.hist(x_gaussian, bins=50, alpha=0.7, density=True, color='gray', edgecolor='black')
ax.plot(x_range, pi_1 * norm.pdf(x_range, mu_1, sigma_1) + 
                  pi_2 * norm.pdf(x_range, mu_2, sigma_2), 'k-', linewidth=2.5, label='True mixture')

ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.set_title('Samples from Gaussian Mixture')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Gaussian Mixture Properties:')
print(f'  Component 1: N(mu={mu_1}, sigma={sigma_1}), weight={pi_1}')
print(f'  Component 2: N(mu={mu_2}, sigma={sigma_2}), weight={pi_2}')

In [ ]:
# Visualize BIC and AIC comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

K_vals = [r['K'] for r in results_gmm]
bic_vals = [r['BIC'] for r in results_gmm]
aic_vals = [r['AIC'] for r in results_gmm]
log_L_vals = [r['log_L'] for r in results_gmm]

# Plot 1: Log-likelihood (always decreases with K)
ax = axes[0]
ax.plot(K_vals, log_L_vals, 'o-', linewidth=2.5, markersize=10, color='green', label='Log-likelihood')
ax.axvline(2, color='red', linestyle='--', alpha=0.5, label='True K=2')
ax.set_xlabel('Number of Components (K)')
ax.set_ylabel('Log-Likelihood')
ax.set_title('(a) Log-Likelihood: Always Improves with More Components')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(K_vals)

# Plot 2: BIC (penalizes complexity more)
ax = axes[1]
ax.plot(K_vals, bic_vals, 's-', linewidth=2.5, markersize=10, color='blue', label='BIC')
ax.scatter([K_vals[best_bic_idx]], [bic_vals[best_bic_idx]], s=200, color='red', 
          marker='*', edgecolor='darkred', linewidth=2, zorder=5, label='BIC minimum')
ax.axvline(2, color='red', linestyle='--', alpha=0.3, label='True K=2')
ax.set_xlabel('Number of Components (K)')
ax.set_ylabel('BIC Score')
ax.set_title('(b) BIC: Prefers Simpler Models')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(K_vals)

# Plot 3: AIC (penalizes complexity less)
ax = axes[2]
ax.plot(K_vals, aic_vals, '^-', linewidth=2.5, markersize=10, color='purple', label='AIC')
ax.scatter([K_vals[best_aic_idx]], [aic_vals[best_aic_idx]], s=200, color='red', 
          marker='*', edgecolor='darkred', linewidth=2, zorder=5, label='AIC minimum')
ax.axvline(2, color='red', linestyle='--', alpha=0.3, label='True K=2')
ax.set_xlabel('Number of Components (K)')
ax.set_ylabel('AIC Score')
ax.set_title('(c) AIC: Allows More Complex Models')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(K_vals)

plt.tight_layout()
plt.show()

print('\nKey Insight:')
print('  - Log-likelihood always decreases (better fit) with more components')
print('  - BIC adds penalty: log(n) * k, prefers simpler models')
print('  - AIC adds penalty: 2 * k, penalizes complexity less')
print('  - BIC correctly identified K=2 as the true model')

In [ ]:
# Model selection: Fit mixtures with different numbers of components
from scipy.optimize import minimize

def fit_gmm_simple(x, K, max_iter=100, tol=1e-4):
    """
    Fit Gaussian Mixture Model with K components using EM algorithm (simplified)
    
    Returns:
      means, variances, weights, log_likelihood, num_params
    """
    N = len(x)
    
    # Initialize
    means = np.linspace(x.min(), x.max(), K)
    variances = np.ones(K) * x.var() / K
    weights = np.ones(K) / K
    
    for iteration in range(max_iter):
        # E-step: Compute responsibilities
        responsibilities = np.zeros((N, K))
        for k in range(K):
            responsibilities[:, k] = weights[k] * norm.pdf(x, means[k], np.sqrt(variances[k]))
        
        # Normalize
        responsibilities /= responsibilities.sum(axis=1, keepdims=True)
        
        # M-step: Update parameters
        N_k = responsibilities.sum(axis=0)
        
        means_new = np.array([np.sum(responsibilities[:, k] * x) / N_k[k] for k in range(K)])
        variances_new = np.array([np.sum(responsibilities[:, k] * (x - means[k])**2) / N_k[k] 
                                  for k in range(K)])
        weights_new = N_k / N
        
        # Check convergence
        if np.max(np.abs(means_new - means)) < tol:
            break
        
        means = means_new
        variances = variances_new
        weights = weights_new
    
    # Compute log likelihood
    likelihoods = np.zeros((N, K))
    for k in range(K):
        likelihoods[:, k] = weights[k] * norm.pdf(x, means[k], np.sqrt(variances[k]))
    
    log_likelihood = np.sum(np.log(likelihoods.sum(axis=1)))
    
    # Number of parameters
    num_params = K * 3 - 1  # K means + K variances + K weights - 1 (constraint)
    
    return means, variances, weights, log_likelihood, num_params

# Test with different numbers of components
print('='*70)
print('Model Selection: Gaussian Mixture Models')
print('='*70)

# Generate data from 2-component mixture
np.random.seed(42)
T = 300
component1 = np.random.normal(-2, 0.5, T//2)
component2 = np.random.normal(2, 0.5, T//2)
x_gmm = np.concatenate([component1, component2])

print(f'\nData: {T} samples from 2-component Gaussian mixture')
print(f'  Component 1: N(μ=-2, σ=0.5)')
print(f'  Component 2: N(μ=2, σ=0.5)')

# Fit models with K = 1, 2, 3, 4, 5
K_values = [1, 2, 3, 4, 5]
results_gmm = []

for K in K_values:
    means, variances, weights, log_L, n_params = fit_gmm_simple(x_gmm, K)
    
    # Compute BIC and AIC
    bic = -2 * log_L + n_params * np.log(T)
    aic = -2 * log_L + 2 * n_params
    
    results_gmm.append({
        'K': K,
        'log_L': log_L,
        'n_params': n_params,
        'BIC': bic,
        'AIC': aic,
        'means': means,
        'variances': variances,
        'weights': weights
    })
    
    print(f'\nK={K} components:')
    print(f'  Log-likelihood: {log_L:10.2f}')
    print(f'  Parameters:     {n_params:10d}')
    print(f'  BIC:            {bic:10.2f} ← {"✓ BEST" if bic == min([r["BIC"] for r in results_gmm]) else ""}')
    print(f'  AIC:            {aic:10.2f} ← {"✓ BEST" if aic == min([r["AIC"] for r in results_gmm]) else ""}')

# Find best model
best_bic_idx = np.argmin([r['BIC'] for r in results_gmm])
best_aic_idx = np.argmin([r['AIC'] for r in results_gmm])

print(f'\n{"="*70}')
print(f'Model Selection Summary:')
print(f'  True model:    K=2')
print(f'  BIC selects:   K={results_gmm[best_bic_idx]["K"]} ✓')
print(f'  AIC selects:   K={results_gmm[best_aic_idx]["K"]}')
print(f'  (BIC prefers simpler model, correctly identifies K=2)')

## Part 8: Model Selection - BIC vs AIC

### The Problem: How Many Components?

```
Questions:
  Should we use 1 component or 2?
  Should we use 2 components or 3?
  
Challenge: More components always fit better!
  (Can always reduce loss by adding parameters)
  
Solution: Penalize model complexity
```

### BIC (Bayesian Information Criterion)

```
BIC = -2 * log(L) + k * log(n)
      
where:
  L = likelihood of best fit
  k = number of parameters
  n = number of observations

Lower BIC → Better model
  (balances fit + simplicity)
```

### AIC (Akaike Information Criterion)

```
AIC = -2 * log(L) + 2k

Similar to BIC but penalizes complexity less
  (AIC prefers slightly more complex models)
```

### When to Use Each

```
BIC: Prefer simpler models, better for selecting true model
AIC: Predict new data, prefer slightly more complex
```

## Summary: Key Concepts

**Mixture Models**:
1. Combine multiple distributions with weights
2. Include hidden/latent variables
3. Can represent complex data with simple components

**Mathematical Form**:
```
p(x) = sum_k: pi_k * p(x | z=k)
```

**Inference**:
- Forward: Given x, infer which component (posterior over z)
- Backward: Given data, learn parameters

**Challenge**: Circular dependency (need parameters to infer states, need states to learn parameters)

**Solution**: EM Algorithm (next section!)